In [1]:
import os
import json
from dotenv import load_dotenv
from typing import Annotated, List, Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage
from langchain_groq import ChatGroq

In [2]:
# ---------------------------------------------------------------------------
# 1. Define the shared State object
# ---------------------------------------------------------------------------
class FlightState(TypedDict):
    messages: Annotated[list, add_messages]  # running chat/log history
    from_place: str
    to_place: str
    available_flights: Optional[List[dict]]  # filled by Node1
    fare_details: Optional[dict]  # filled by Node2
    booking_status: Optional[dict]

In [4]:
load_dotenv()

True

In [5]:
# ---------------------------------------------------------------------------
# 2. Initialize the LLM
# ---------------------------------------------------------------------------
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
)

In [6]:
# ---------------------------------------------------------------------------
# 3. Node 1 - Search available flights (with names)
# ---------------------------------------------------------------------------
def search_flights(state: FlightState) -> FlightState:
    prompt = (
        f"Generate 3 realistic sample flights from {state['from_place']} to "
        f"{state['to_place']}. Respond ONLY as JSON list, each item having "
        f"keys: flight_name, flight_number, departure_time, arrival_time."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("==========Node-1 LLM RESPONSE ==========")
    print(response.content)
    print("==================================")
    try:
        flights = json.loads(response.content)
        print("✅ JSON Parse Success")
    except json.JSONDecodeError:
        # Fallback mock data if the model doesn't return clean JSON
        print("❌ JSON Parse Failed")
        flights = [
            {
                "flight_name": "IndiGo",
                "flight_number": "6E-201",
                "departure_time": "09:00",
                "arrival_time": "11:00",
            },
            {
                "flight_name": "Air India",
                "flight_number": "AI-402",
                "departure_time": "13:30",
                "arrival_time": "15:45",
            },
            {
                "flight_name": "Vistara",
                "flight_number": "UK-955",
                "departure_time": "18:15",
                "arrival_time": "20:20",
            },
        ]

    return {
        "messages": [AIMessage(content=f"Found {len(flights)} flights.")],
        "available_flights": flights,
    }

In [7]:
#test
test_state = {
    "messages": [],
    "from_place": "Hyderabad",
    "to_place": "Delhi",
    "available_flights": None,
    "fare_details": None,
    "booking_status": None,
}

result = search_flights(test_state)

print("\nReturned Result:")
print(result)

==========Node-1 LLM RESPONSE ==========
[
  {
    "flight_name": "IndiGo",
    "flight_number": "6E201",
    "departure_time": "06:00",
    "arrival_time": "08:05"
  },
  {
    "flight_name": "Air India",
    "flight_number": "AI501",
    "departure_time": "11:30",
    "arrival_time": "13:35"
  },
  {
    "flight_name": "SpiceJet",
    "flight_number": "SG123",
    "departure_time": "18:45",
    "arrival_time": "20:50"
  }
]
✅ JSON Parse Success

Returned Result:
{'messages': [AIMessage(content='Found 3 flights.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'available_flights': [{'flight_name': 'IndiGo', 'flight_number': '6E201', 'departure_time': '06:00', 'arrival_time': '08:05'}, {'flight_name': 'Air India', 'flight_number': 'AI501', 'departure_time': '11:30', 'arrival_time': '13:35'}, {'flight_name': 'SpiceJet', 'flight_number': 'SG123', 'departure_time': '18:45', 'arrival_time': '20:50'}]}


In [8]:
# ---------------------------------------------------------------------------
# 4. Node 2 - Check fare, from and to places
# ---------------------------------------------------------------------------

class FareInfo(TypedDict):
    from_place: str
    to_place: str
    cheapest_flight: str
    fare_inr: int

structured_llm = llm.with_structured_output(FareInfo)

def check_fare_and_route(state: FlightState) -> FlightState:
    prompt = (
        f"For a flight from {state['from_place']} to {state['to_place']}, "
        f"given these flights: {state['available_flights']}. "
        f"Respond ONLY as JSON with keys: from_place, to_place, "
        f"cheapest_flight, fare_inr (a realistic integer amount)."
    )
    fare_info  = structured_llm.invoke([HumanMessage(content=prompt)])
    print("==========Node-2 LLM RESPONSE ==========")
    print(fare_info )
    print("==================================")
    return {
        "messages": [AIMessage(content=f"Fare checked: {fare_info}")],
        "fare_details": fare_info,
    }

In [9]:
# Test
test_state = {
    "messages": [],
    "from_place": "Hyderabad",
    "to_place": "Delhi",
    "available_flights": None,
    "fare_details": None,
    "booking_status": None,
}

# First Node
state_after_node1 = search_flights(test_state)

# from_place aur to_place bhi chahiye, kyunki node1 return me ye nahi aa rahe
state_after_node1["from_place"] = test_state["from_place"]
state_after_node1["to_place"] = test_state["to_place"]

# Second Node
result = check_fare_and_route(state_after_node1)

print(result)

==========Node-1 LLM RESPONSE ==========
[
  {
    "flight_name": "IndiGo",
    "flight_number": "6E201",
    "departure_time": "06:00",
    "arrival_time": "08:10"
  },
  {
    "flight_name": "Air India",
    "flight_number": "AI501",
    "departure_time": "11:30",
    "arrival_time": "13:40"
  },
  {
    "flight_name": "SpiceJet",
    "flight_number": "SG123",
    "departure_time": "18:45",
    "arrival_time": "20:55"
  }
]
✅ JSON Parse Success
==========Node-2 LLM RESPONSE ==========
{'cheapest_flight': 'SpiceJet', 'fare_inr': 3500, 'from_place': 'Hyderabad', 'to_place': 'Delhi'}
{'messages': [AIMessage(content="Fare checked: {'cheapest_flight': 'SpiceJet', 'fare_inr': 3500, 'from_place': 'Hyderabad', 'to_place': 'Delhi'}", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'fare_details': {'cheapest_flight': 'SpiceJet', 'fare_inr': 3500, 'from_place': 'Hyderabad', 'to_place': 'Delhi'}}


In [10]:
# ---------------------------------------------------------------------------
# 5. Node 3 - Booking confirmation
# ---------------------------------------------------------------------------
def booking_confirmation(state: FlightState) -> FlightState:
    fare = state["fare_details"]
    booking = {
        "status": "CONFIRMED",
        "flight_booked": fare["cheapest_flight"],
        "from_place": fare["from_place"],
        "to_place": fare["to_place"],
        "amount_paid_inr": fare["fare_inr"],
        "pnr": "PNR" + str(abs(hash(fare["cheapest_flight"])) % 100000),
    }
    return {
        "messages": [AIMessage(content=f"Booking confirmed: {booking}")],
        "booking_status": booking,
    }

In [11]:
# ---------------------------------------------------------------------------
# 6. Build the StateGraph
# ---------------------------------------------------------------------------
graph_builder = StateGraph(FlightState)

graph_builder.add_node("search_flights", search_flights)
graph_builder.add_node("check_fare_and_route", check_fare_and_route)
graph_builder.add_node("booking_confirmation", booking_confirmation)

graph_builder.add_edge(START, "search_flights")
graph_builder.add_edge("search_flights", "check_fare_and_route")
graph_builder.add_edge("check_fare_and_route", "booking_confirmation")
graph_builder.add_edge("booking_confirmation", END)

graph = graph_builder.compile()

In [12]:
# ---------------------------------------------------------------------------
# 7. Run the graph and print the full final state
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    initial_state: FlightState = {
        "messages": [HumanMessage(content="Book a flight from Hyderabad to USA")],
        "from_place": "Hyderabad",
        "to_place": "USA",
        "available_flights": None,
        "fare_details": None,
        "booking_status": None,
    }

    final_state = graph.invoke(initial_state)

    print("\n================ FINAL STATE OBJECT ================\n")
    print(
        json.dumps(
            {
                "from_place": final_state["from_place"],
                "to_place": final_state["to_place"],
                "available_flights": final_state["available_flights"],
                "fare_details": final_state["fare_details"],
                "booking_status": final_state["booking_status"],
            },
            indent=2,
        )
    )

    print("\n================ MESSAGE LOG ================\n")
    for m in final_state["messages"]:
        print(f"[{m.type}] {m.content}")

==========Node-1 LLM RESPONSE ==========
[
  {
    "flight_name": "Air India",
    "flight_number": "AI 127",
    "departure_time": "01:30",
    "arrival_time": "06:35"
  },
  {
    "flight_name": "Lufthansa",
    "flight_number": "LH 756",
    "departure_time": "02:15",
    "arrival_time": "13:10"
  },
  {
    "flight_name": "American Airlines",
    "flight_number": "AA 292",
    "departure_time": "22:45",
    "arrival_time": "05:30"
  }
]
✅ JSON Parse Success
==========Node-2 LLM RESPONSE ==========
{'cheapest_flight': 'Air India', 'fare_inr': 50000, 'from_place': 'Hyderabad', 'to_place': 'USA'}

================ FINAL STATE OBJECT ================

{
  "from_place": "Hyderabad",
  "to_place": "USA",
  "available_flights": [
    {
      "flight_name": "Air India",
      "flight_number": "AI 127",
      "departure_time": "01:30",
      "arrival_time": "06:35"
    },
    {
      "flight_name": "Lufthansa",
      "flight_number": "LH 756",
      "departure_time": "02:15",
      "arrival

In [ ]:
# Custom function
from langchain_core.tools import tool
@ tool
def multiply(a:int,b:int)->int:
    """Multiply a and b
    Args:
        a (int): first int
        b (int): second int

    Returns:
        int: output int
    """
    return a*b

tools=[multiply]
llm_with_tool=llm.bind_tools(tools)

In [21]:
from langchain_core.tools import tool
from langchain_groq import ChatGroq


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b


tools = [multiply]

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(
    "Multiply 5 and 6. Use the multiply tool."
)

print(response)
print(response.tool_calls)

content='' additional_kwargs={'tool_calls': [{'id': 'n556c6j29', 'function': {'arguments': '{"a":5,"b":6}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 224, 'total_tokens': 243, 'completion_time': 0.039090186, 'completion_tokens_details': None, 'prompt_time': 0.017126825, 'prompt_tokens_details': None, 'queue_time': 0.082265174, 'total_time': 0.056217011}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fcf64-4c2e-78b1-905c-db2db2b375d5-0' tool_calls=[{'name': 'multiply', 'args': {'a': 5, 'b': 6}, 'id': 'n556c6j29', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 224, 'output_tokens': 19, 'total_tokens': 243}
[{'name': 'multiply', 'args': {'a': 5, 'b': 6}, 'id': 'n556c6j29', 'type': 'tool_call'}]


In [22]:
tool_call = response.tool_calls[0]

result = multiply.invoke(tool_call["args"])

print(result)

30


In [23]:
print("Tool result:", result)

Tool result: 30
